## __`Speech Command Recognition`__ 

__NOM ETUDIANT : Gérard BENITAH__

lauriane.bompay@irit.fr

:::
__Présentation de l'article__.   
L’article présente un système de reconnaissance de commandes vocales, à faible latence et faible empreinte mémoire, basé sur différents modèles de réseaux de neurones (softmax simple, DNN et CNN) appliqués au Speech Commands Dataset de Google (65 000 fichiers WAV, 30 mots, 1 s chacun).
Les auteurs extraient des caractéristiques MFCC sur des fenêtres de 30 ms (stride 10 ms), produisant des matrices 2D 98×40 qui servent d’entrée aux réseaux.

:::
:::
__Abstract__
Projet visant à construire un système de reconnaissance de commandes vocales précis, léger et à faible latence, capable de détecter des mots-clés prédéfinis.
À partir du jeu de données Speech Commands de Google, différentes architectures sont testées : 
 - modèle softmax simple, 
 - réseau de neurones profond (DNN) et 
 - réseau de neurones convolutif (CNN).
Le CNN surpasse les deux autres modèles et atteint une exactitude de 95,1% sur 6 étiquettes.
:::
:::
__Résumée de l’introduction__.  
L’interaction vocale sur mobile (Google Now, Siri, “Ok Google”) repose sur la détection de mots-clés (keyword spotting, KWS), pratique pour les scénarios mains libres et les situations de conduite ou d’urgence.
Comme ces systèmes tournent sur smartphone ou tablette, ils doivent être rapides, peu gourmands en mémoire et en calcul, d’où la recherche d’architectures compactes.
Le système classifie chaque clip d’une seconde comme « silence », « mot inconnu » ou l’un des mots-clés prédéfinis (par ex. yes/no, up/down/left/right, stop/go), puis utilise les trois modèles (softmax, DNN, CNN) pour produire une probabilité et un label final.
:::
:::
__Résumée de la partie modèles et résultats__.   
 - `Vanilla softmax` : une seule couche entièrement connectée avec softmax, très rapide mais peu précise (environ 56% de précision test).

 - `DNN` : 3 couches cachées de 128 neurones avec ReLU et dropout, meilleure précision (≈72%) mais plus de paramètres et de calcul.

 - `CNN` : 2 couches de convolution, couches de pooling, dropout, couche linéaire basse dimension, puis fully-connected + softmax, moins de 250 k paramètres et meilleure performance (≈94,5% test, meilleure précision, rappel, AUC).
:::

​

## `I Vanilla Softmax`

In [1]:
#%pip install numpy scipy matplotlib tqdm librosa
#%pip install numpy scipy matplotlib tqdm librosa
#python -m pip install ipykernel
#python -m ipykernel install --user --name tap_env --display-name "Python (tap_env)"

In [1]:
import torch, torchaudio
print(torch.__version__, torchaudio.__version__)

from torchaudio.datasets import SPEECHCOMMANDS

2.5.1 2.5.1


### `1.1 Technologie de repérage de mots-clés (KWS)`

:::
`Technologie de repérage de mots-clés (KWS) fournie une interface mains libres: `
 __`“yes”/“no” ou “up”/“down”/“left”/“right” ou “stop”/“go”`.__ 
 
 - _KEYWORDS = ["up", "down",“yes”,“no”,...........,“left”,“right”,“stop”,“go”,"left", "right"]_
 - _SILENCE_LABEL = "_silence_"_
 - _UNKNOWN_LABEL = "_unknown_" #  the background volume is 0.1 and its frequency is 0.8._
:::

In [5]:
# construction du mapping avec 2 labels spéciaux d'abord
SILENCE_LABEL = "_silence_"
UNKNOWN_LABEL = "_unknown_"

KEYWORDS=[]

label2id = {
    SILENCE_LABEL: 0,
    UNKNOWN_LABEL: 1,
}

def _word_to_label(word: str) -> str:
    if word == SILENCE_LABEL:
        return SILENCE_LABEL
    if word in KEYWORDS:
        return word
    return UNKNOWN_LABEL  

### `1.2 Chargement des WAV et extraction MFCC`
:::
 - parcourt une arborescence de WAV (type Speech Commands) ;
 - convertit chaque fichier en MFCC fixes 98×40 ;
 - remappe les dossiers/mots en labels compacts silence / keywords / unknown.

 <p style="text-align:left;">
<img src="/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_TAP/TP1_2026/image/mfcc.png" alt="Diagram of MFCC Derivation Process" style="width:80%; height:auto;" title="Titre optionnel" />
</p>

 - Utilisation torchaudio.transforms.MFCC, qui encapsule : 
 - pré‑accentuation → fenêtrage → FFT → banc de filtres Mel → log → DCT → Mel Cepstrum / MFCC
:::
:::
`SPEECHCOMMANDS retourne un dataset contenant: `
   - __waveform__ : tenseur audio shape=[1,T], les échantillons du fichier WAV.
   - __sample_rate__ : entier (en général 16000 Hz).
   - __label__ : chaîne de caractères avec le mot prononcé (nom du dossier : "up", "down", "yes", etc.).
   - __speaker_id__ : identifiant texte de la personne qui parle.
   - __utterance_number__ : numéro de l’énoncé pour ce locuteur (entier).
Le dataset encapsule les fichiers + les métadonnées, charge le waveform à l'on appel de  __getitem__ (dataset[i]).
::: 

In [3]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchaudio

class SpeechCommandsMFCCDataset(Dataset):
    def __init__(
            self, root, subset=None, sample_rate=16000,
            n_mfcc=40, win_length_ms=30, hop_length_ms=10,target_T=98):
        self.sample_rate = sample_rate
        self.target_T = target_T

        # Dataset de base torchaudio
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=root,
            download=True,
            subset=subset  # "training", "validation", "testing" ou None
        )

        # Paramètres MFCC en échantillons
        win_length = int(sample_rate * win_length_ms / 1000)
        hop_length = int(sample_rate * hop_length_ms / 1000)

        self.mfcc_transform = torchaudio.transforms.MFCC(
            sample_rate=sample_rate,
            n_mfcc=n_mfcc,
            melkwargs = {
                "win_length": win_length,
                "hop_length": hop_length,
                "center": True,
                "f_min": 20.0,
                "n_mels": 40,
                "n_fft": 512,
                "f_max": 8000.0,
            },
        )

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        waveform, sr, label_str, speaker_id, utt = self.base[idx]  # waveform: (1, T)
        #print(f'waveform{waveform}\nsr : {sr} label_str{label_str} speaker_id{speaker_id} utt:{utt}')

        # Resample si besoin
        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(
                waveform, sr, self.sample_rate
            )

        # MFCC: (1, n_mfcc, T)
        mfcc = self.mfcc_transform(waveform)
        mfcc = mfcc.squeeze(0)                  # (n_mfcc, T)

        # Padding / troncature à target_T
        T = mfcc.shape[1]
        #print(f'avant nombre de frame {T}')
        if T < self.target_T:
            pad = self.target_T - T
            mfcc = F.pad(mfcc, (0, pad))       # (n_mfcc, target_T)
        else:
            mfcc = mfcc[:, :self.target_T]
            #print(f'après nombre de frame {mfcc.shape[1]}')

                # Remapping du label texte -> label compact 0..5        
        mapped = _word_to_label(label_str)            # "_silence_", "up", ..., "_unknown_"
        label_id = label2id[mapped]                   # entier 0..5    
        
        # (T, n_mfcc) = (98, 40)
        mfcc = mfcc.transpose(0, 1)
        #print(f'end mfcc: {mfcc.shape} label_str: {label_str} label_id: {label_id}   mapped:  {mapped}')

        return mfcc, label_id


### `1.3. Dataset et DataLoader`

In [4]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

root_dir = "M2_IAFA/TP_M2IAFA/TP_M2_TAP/TP1_2026/data/speech_commands/"

dataset = SpeechCommandsMFCCDataset(
            root_dir, 
            subset=None, 
            sample_rate=16000,
            n_mfcc=40, 
            win_length_ms=30, 
            hop_length_ms=10,
            target_T=98
            )

n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)

n_test  = n_total - n_train - n_val 

generator = torch.Generator().manual_seed(42)  # pour un split reproductible

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=generator,
)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=100, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=100, shuffle=False)

### `1.4 Identification de classes`
:::
 - _KEYWORDS = ["up", "down",“yes”,“no”,...........,“left”,“right”,“stop”,“go”,"left", "right"]_
 - _SILENCE_LABEL = "_silence_"_
 - _UNKNOWN_LABEL = "_unknown_"_
:::


In [6]:
from pathlib import Path

root = Path(root_dir)
count = sum(1 for _ in root.rglob("*.wav"))
print("Nombre de fichiers .wav :", count)

Nombre de fichiers .wav : 211670


In [7]:
labels=[]
nb_occ = len(dataset)
for i in range(nb_occ):
    labels.append(dataset.base.get_metadata(i)[2])

KEYWORDS = list(set(labels))            # liste des labels
print(f"nombre d'occurences : {nb_occ}")    

nombre d'occurences : 105829


In [8]:
dataset.base.get_metadata(105828)

('speech_commands_v0.02/zero/fffcabd1_nohash_0.wav',
 16000,
 'zero',
 'fffcabd1',
 0)

In [9]:
from collections import Counter

classes = set(labels)
nbre_classes = len(classes)
print("Nombre de classes :", nbre_classes)

Nombre de classes : 35


In [10]:
for i, lab in enumerate(classes):
    label2id[lab] = i+2      

print(f"label2id            : {label2id}")            

label2id            : {'_silence_': 0, '_unknown_': 1, 'backward': 2, 'seven': 3, 'on': 4, 'bird': 5, 'wow': 6, 'yes': 7, 'marvin': 8, 'go': 9, 'no': 10, 'cat': 11, 'six': 12, 'three': 13, 'sheila': 14, 'four': 15, 'learn': 16, 'nine': 17, 'right': 18, 'five': 19, 'forward': 20, 'up': 21, 'tree': 22, 'house': 23, 'down': 24, 'off': 25, 'two': 26, 'left': 27, 'follow': 28, 'happy': 29, 'visual': 30, 'eight': 31, 'one': 32, 'bed': 33, 'dog': 34, 'zero': 35, 'stop': 36}


### `1.5 Modèle VanillaSoftMax`

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VanillaSoftmaxSpeech(nn.Module):
    def __init__(self, n_frames=98, n_mfcc=40, n_classes=37):
        super().__init__()
        self.in_features = n_frames * n_mfcc  # 98 * 40 = 3920
        self.fc = nn.Linear(self.in_features, n_classes)

    def forward(self, x):                # x peut être (batch, 1, 98, 40) ou (batch, 98, 40)
        #print(f'x.shape : {x.shape}')
        if x.dim() == 4:                  # (B, C, T, F) -> (B, T, F) si C=1
            x = x.squeeze(1)
        # Aplatissement en vecteur
        x = x.reshape(x.size(0), -1)      # (B, 3920)
        logits = self.fc(x)               # (B, 30)
        return logits                     # CrossEntropyLoss fera softmax

### `1.6 Instanciation de VanillaSoftMax`

`Avec nn.CrossEntropyLoss sur une classification à C classes`
​Une loss proche de log(C) correspond à un modèle qui devine au hasard pour C=6, log(6)≈0,77815125

In [12]:
model = VanillaSoftmaxSpeech(n_frames=98, n_mfcc=40, n_classes=37)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### `1.7 Phase d'apprentissage du modèle`

In [19]:
def train_model(model, data_loader, criterion, epoch=1):
    for epoch in range(epoch):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (mfcc, labels) in enumerate(data_loader):
            optimizer.zero_grad()
            logits = model(mfcc)                    # mfcc: (B, 1, 98, 40) ou (B, 98, 40)  
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            # accumulate loss
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size

            # accumulate accuracy
            preds = logits.argmax(dim=1) 
            correct += (preds == labels).sum().item()

            #if i % 10 == 0: 
            #   print("train batch loss:", loss.item())
            #   print("train preds != labels:", (preds != labels).sum().item())
            

        train_loss = running_loss / total
        train_acc = correct / total
        print(f"Epoch {epoch+1}: train loss = {train_loss:.4f}, "  f"train acc = {train_acc*100:.2f}%")    

In [20]:
train_model(model, train_loader, criterion, epoch=2)

Epoch 1: train loss = 11.0367, train acc = 30.00%


KeyboardInterrupt: 

### `1.8 Phase d'évaluation du modèle`

In [11]:
import torch.nn.functional as F

def evaluate(model, data_loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for i, (mfcc, labels) in enumerate(data_loader):
            logits = model(mfcc)
            loss = criterion(logits, labels)

            # accumulate loss
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size

            #if i == 5 : 
              #  break

            # accuracy
            preds = logits.argmax(dim=1)      # pas besoin de softmax pour argmax
            correct += (preds == labels).sum().item()

            #print("val batch loss:", loss.item())
            #print("preds != labels:", (preds != labels).sum().item())

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [12]:
val_loss, val_acc = evaluate(model, val_loader, criterion)
print(f"Val loss = {val_loss:.4f}, Val acc = {val_acc*100:.2f}%")

Val loss = 13.0635, Val acc = 99.94%


### `1.9 Phase de test du modèle`

In [13]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Val loss = {test_loss:.4f}, Val acc = {test_acc*100:.2f}%")

Val loss = 14.7813, Val acc = 99.91%


:::
`Lien avec la CrossEntropy`
La loss par exemple vaut : 
- $\small log(p_{true})$, où $p_{true}$ est la sortie softmax associée au label correct
- si $p_{true}$ est proche de 1, la loss est proche de 0
- si $p_{true}$ est très petite (quasi 0), la loss devient très grande
__Interprétation dans tes logs__.   
 - Lorsque la loss est importante pour un seul exemple, $p_{true}$ est quasiment nul pour ce point, 
   - cela signifie que le modèle est très sûr de la mauvaise classe.
 - Lorsque tous les batchs ont une loss 0, 
   - cela correspond au cas (numérique) où $p_{true}$≈1 partout, donc CrossEntropy ≈ 0.
:::   

:::
Décommenter les traces: 
 - _train batch loss: 32.060508728027344._   
 - _train preds != labels: 64._  
 - _train batch loss: 0.0._  
 - _._._  
 - _._._  
 - _train batch loss: 722.9816284179688._  
 - _train preds != labels: 1_._  
 - _Epoch 1: train loss = 125.8404, train acc = 83.07%._ 
:::      

## `2. Deep Neuronal Network`
:::
L'Architecture du second modèle s'apppuie sur un réseau neuronal standard _fully connected_ avec 3 couches cachées et 128 nœuds cachés par couche utilisant la fonction d'activation RelU et du dropout Ce choix est du au fait que le RNN avec 3 hidden layers fully connected surpasse ceux à 1 ou 2, et qui se dégrade avec 4 ou plus. 

<p style="text-align:center;">
<img src="/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_TAP/TP1_2026/image/DNN.png" alt="Diagram of MFCC Derivation Process" style="width:30%; height:auto;" title="Titre optionnel" />
</p>
:::
:::
 - L'augmentation du nombre de nœuds dans les couches cachées permet d'améliorer la précision, au détriment de risque d'overfitting, qui est compensé par la mise en place du dropout.

 - La fonction d'activation permet de briser la linéarité entre couche et le choix ReLU de limiter les problèmes de gradient tout en conservant une bonne capacité de modélisation.

 - Comparé à Vanilla Single Layer, ce modèle devrait donner un résultat plus précis au prix d'une plus grande empreinte mémoire et d'un coût de calcul plus élevé. 
:::

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpeechDNN(nn.Module):
    def __init__(self, input_dim=98*40, hidden_dim=128, num_classes=37, dropout_p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        # x: (B, 98, 40) ou (B, 1, 98, 40)
        if x.dim() == 4:
            x = x.squeeze(1)        # (B, 98, 40)
        x = x.reshape(x.size(0), -1)  # (B, 3920)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        logits = self.out(x)         # (B, num_classes)
        return logits


In [ ]:
# vérification du nombre de paramètres
model = SpeechDNN(input_dim=98*40, hidden_dim=128, num_classes=37, dropout_p=0.5)
n_params = sum(p.numel() for p in model.parameters())
print("Nb de paramètres :", n_params)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [16]:
train_model(model, train_loader, criterion, epoch=2)

Epoch 1: train loss = 0.0085, train acc = 99.89%
Epoch 2: train loss = 0.0000, train acc = 100.00%


In [17]:
val_loss, val_acc = evaluate(model, val_loader, criterion)
print(f"Val loss = {val_loss:.4f}, Val acc = {val_acc*100:.2f}%")

Val loss = 0.0000, Val acc = 100.00%


In [18]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Val loss = {test_loss:.4f}, Val acc = {test_acc*100:.2f}%")

Val loss = 0.0000, Val acc = 100.00%


## `III Convolution Neural Network CNN`

<p style="text-align:left;">
<img src="/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_TAP/TP1_2026/image/cnn.png" alt="Diagram of MFCC Derivation Process" style="width:80%; height:auto;" title="Titre optionnel" />
</p>

Voici un modèle PyTorch avec 2 convolutions, pooling, dropout, une couche linéaire de petite dimension, puis fully‑connected + softmax, avec un ordre de grandeur < 250 k paramètres pour des entrées MFCC (B,1,98,40).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpeechCommandsCNN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()

        # Entrée: (B, 1, 98, 40)

        # Conv1: 64 filtres, noyau 5x5 par ex. -> (B, 64, 98, 40)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, padding=2)

        # MaxPool 1x3 (fig) uniquement sur la dimension fréquence (40 -> ~13)
        self.pool = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        # Conv2: 64 filtres, noyau 5x5 -> (B, 64, 98, 13)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, padding=2)

        self.dropout = nn.Dropout(0.5)

        # Calcul de la taille après conv/pool:
        # entrée (98, 40)
        # après pool (1x3) -> (98, 13) ~ "10x4x64" du schéma
        # donc flatten_dim = 64 * 98 * 13
        self.flatten_dim = 64 * 98 * 13

        # Lin 32 (couche basse dimension)
        self.fc1 = nn.Linear(self.flatten_dim, 32)

        # FC 128
        self.fc2 = nn.Linear(32, 128)

        # FC sortie (37 classes)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (B, 1, 98, 40)
        x = F.relu(self.conv1(x))
        x = self.pool(x)              # (B, 64, 98, 13)
        x = self.dropout(x)

        x = F.relu(self.conv2(x))     # (B, 64, 98, 13)
        x = self.dropout(x)

        x = x.view(x.size(0), -1)     # flatten

        x = F.relu(self.fc1(x))       # 32
        x = self.dropout(x)

        x = F.relu(self.fc2(x))       # 128
        x = self.dropout(x)

        logits = self.fc3(x)          # (B, 37)
        return logits                 # softmax dans CrossEntropyLoss

# exemple d'utilisation
model = SpeechCommandsCNN(num_classes=37)
dummy = torch.randn(8, 1, 98, 40)
logits = model(dummy)
print(logits.shape)   # torch.Size([8, 37])


In [ ]:
# vérification du nombre de paramètres
model = SimpleCNN(num_classes=6)
n_params = sum(p.numel() for p in model.parameters())
print("Nb de paramètres :", n_params)